# Baseline: Caser

Tang & Wang (2018). Recomendación secuencial con redes convolucionales sobre el historial.

En local se ejecuta desde `notebooks/` y lee los datos desde `../repo_latentr3/data`. En Colab monta Google Drive y lee los datos desde `data_amazon_2018/`.

## Importaciones

In [ ]:
import os
import ast
import random
import pandas as pd
import numpy as np
import torch
import torch.nn as nn

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device, torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

## Carga de datos

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False


In [ ]:
base = 'data_amazon_2018' if IN_COLAB else os.path.join('..', 'repo_latentr3', 'data')
names = {"CDs_and_Vinyl": "cd_data",
         "Musical_Instruments": "music_data",
         "Toys_and_Games": "toys_data",
         "Video_Games": "games_data"}
list_cols = ["item_asins", "history_item_id", "history_item_title", "history_rating", "history_timestamp"]

def get_name(nombre_archivo):
    for categoria, nombre in names.items():
        if nombre_archivo.startswith(categoria):
            return nombre

def add_data_csv(dir, data_dict):
    for nombre_archivo in os.listdir(dir):
        df = pd.read_csv(os.path.join(dir, nombre_archivo))
        for col in list_cols:
            df[col] = df[col].apply(ast.literal_eval)
        data_dict[get_name(nombre_archivo)] = df

def add_data_info(dir, data_dict):
    for nombre_archivo in os.listdir(dir):
        filas = []
        with open(os.path.join(dir, nombre_archivo), encoding="utf-8") as f:
            for linea in f:
                title, item_id = linea.rstrip("\n").rsplit("\t", 1)
                filas.append((title, int(item_id)))
        data_dict[get_name(nombre_archivo)] = pd.DataFrame(filas, columns=["title", "item_id"])

In [ ]:
info, train, valid, test = {}, {}, {}, {}
add_data_info(os.path.join(base, "info"), info)
add_data_csv(os.path.join(base, "train"), train)
add_data_csv(os.path.join(base, "valid"), valid)
add_data_csv(os.path.join(base, "test"), test)
n_items = {nombre: len(df) for nombre, df in info.items()}
print(n_items)

## Métricas

In [ ]:
def hit(recs, objetivo, k):
    return 1 if objetivo in list(recs[:k]) else 0

def ndcg(recs, objetivo, k):
    recs = list(recs[:k])
    if objetivo not in recs:
        return 0
    pos = recs.index(objetivo)
    return 1 / np.log2(pos + 2)

def evaluar(recomendaciones, test_df, ks=[5, 10]):
    resultados = {}
    for k in ks:
        resultados[f"HR@{k}"] = np.mean([hit(r, o, k) for r, o in zip(recomendaciones, test_df["item_id"])])
        resultados[f"NDCG@{k}"] = np.mean([ndcg(r, o, k) for r, o in zip(recomendaciones, test_df["item_id"])])
    return resultados

## Modelo Caser

## Evaluación

Cada modelo produce, por cada fila de `test`, una lista ordenada de `item_id` sobre todo el catálogo. Métricas: HR y NDCG @5 y @10, comparables con la Tabla 1 del paper.

In [ ]:
filas = []
resultados = pd.DataFrame(filas).round(4)
resultados